# Train Models
This notebook loads the data, extracts features using `utils.py`, and trains the simple and improved models.

In [15]:
import os
import pandas as pd
import sklearn.model_selection
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib
from utils import extract_simple_features, extract_improved_features, extract_lluc_features, extract_final_features

## Data Loading and Splitting

In [16]:
quora_df = pd.read_csv("./quora_data.csv")

A_df, test_df = sklearn.model_selection.train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = sklearn.model_selection.train_test_split(A_df, test_size=0.05, random_state=123)

print('train_df.shape=', train_df.shape)
print('val_df.shape=', val_df.shape)
print('test_df.shape=', test_df.shape)

train_df.shape= (291897, 6)
val_df.shape= (15363, 6)
test_df.shape= (16172, 6)


## Simple Model
Using simple word overlap as feature.

In [17]:
X_train_simple = extract_simple_features(train_df)
X_val_simple = extract_simple_features(val_df)
y_train = train_df['is_duplicate'].values
y_val = val_df['is_duplicate'].values

model_simple = LogisticRegression()
model_simple.fit(X_train_simple, y_train)
print(f"Simple Model Accuracy on Val: {model_simple.score(X_val_simple, y_val):.4f}")

Simple Model Accuracy on Val: 0.6458


## Improved Model
Using word overlap, Jaccard similarity, and length differences.

In [18]:
X_train_improved = extract_improved_features(train_df)
X_val_improved = extract_improved_features(val_df)

model_improved = LogisticRegression()
model_improved.fit(X_train_improved, y_train)
print(f"Improved Model Accuracy on Val: {model_improved.score(X_val_improved, y_val):.4f}")

Improved Model Accuracy on Val: 0.6622


## Lluc's Model
Using first word check, WH word check, along with simple features and a Decision Tree.

In [19]:
X_train_lluc = extract_lluc_features(train_df)
X_val_lluc = extract_lluc_features(val_df)

model_lluc = DecisionTreeClassifier(max_depth=5, random_state=123)
model_lluc.fit(X_train_lluc, y_train)
print(f"Lluc's Model Accuracy on Val: {model_lluc.score(X_val_lluc, y_val):.4f}")

Lluc's Model Accuracy on Val: 0.6311


## Final Combined Model
Combines all features: Base, Miki's, Jose's, and Lluc's into a Random Forest Classifier.

In [20]:
X_train_final = extract_final_features(train_df)
X_val_final = extract_final_features(val_df)

model_final = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=123, n_jobs=-1)
model_final.fit(X_train_final, y_train)
print(f"Final Model Accuracy on Val: {model_final.score(X_val_final, y_val):.4f}")

Final Model Accuracy on Val: 0.7202


## Save Models
Save the trained models to the `models` folder.

In [21]:
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/model_simple.pkl'):
    joblib.dump(model_simple, 'models/model_simple.pkl')
if not os.path.exists('models/model_improved.pkl'):
    joblib.dump(model_improved, 'models/model_improved.pkl')
if not os.path.exists('models/model_lluc.pkl'):
    joblib.dump(model_lluc, 'models/model_lluc.pkl')
if not os.path.exists('models/model_final.pkl'):
    joblib.dump(model_final, 'models/model_final.pkl')
print('Models saved successfully.')

Models saved successfully.
